In [91]:
import torch

In [92]:
word_embedding_tensor = torch.tensor([
    [0.8, 1.2, 0.3, 1.7],
    [1.1, 0.6, 0.9, 1.4],
    [0.5, 1.3, 0.7, 1.0]
], dtype=torch.float32)

In [93]:
def positional_encoding(seq_len, d_model):
    position = torch.arange(seq_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-torch.log(torch.tensor(10000.0)) / d_model))
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

# Get dimensions
seq_len, d_model = word_embedding_tensor.shape

# Calculate positional encoding
pos_encoding = positional_encoding(seq_len, d_model)
print("Positional Encoding Tensor:")
print(pos_encoding)
print()

# Add positional encoding to word embeddings
embeddings_with_pos = word_embedding_tensor + pos_encoding
print("Embeddings + Positional Encoding:")
print(embeddings_with_pos)
print()

Positional Encoding Tensor:
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0100,  0.9999],
        [ 0.9093, -0.4161,  0.0200,  0.9998]])

Embeddings + Positional Encoding:
tensor([[0.8000, 2.2000, 0.3000, 2.7000],
        [1.9415, 1.1403, 0.9100, 2.4000],
        [1.4093, 0.8839, 0.7200, 1.9998]])



In [94]:
# Query Weight (W^T)
query_weight = torch.tensor([
    [0.5, 0.3, 0.7, 0.2],
    [0.8, 0.1, 0.5, 0.2],
    [0.3, 0.8, 0.9, 0.5],
    [0.5, 0.8, 0.9, 0.2]
])

# Query Bias (b)
query_bias = torch.tensor([
    [0.2, 0.7, 0.8, 0.8],
    [0.2, 0.7, 0.8, 0.8],
    [0.2, 0.7, 0.8, 0.8]
])

# Key Weight (W^T)
key_weight = torch.tensor([
    [0.2, 0.7, 0.2, 0.5],
    [0.8, 0.8, 0.2, 0.8],
    [0.5, 0.3, 0.8, 0.8],
    [0.2, 0.9, 0.2, 0.7]
])

# Key Bias (b)
key_bias = torch.tensor([
    [0.5, 0.5, 0.8, 0.2],
    [0.5, 0.5, 0.8, 0.2],
    [0.5, 0.5, 0.8, 0.2]
])

# Value Weight (W^T)
value_weight = torch.tensor([
    [0.07, 0.33, 0.03, 0.37],
    [0.12, 0.28, 0.18, 0.22],
    [0.27, 0.13, 0.23, 0.17],
    [0.38, 0.04, 0.32, 0.06]
])

# Value Bias (b)
value_bias = torch.tensor([
    [0.28, 0.47, 0.72, 0.95],
    [0.28, 0.47, 0.72, 0.95],
    [0.28, 0.47, 0.72, 0.95]
])

In [95]:
# Calculate Q, K, V matrices
# Q = (X + PE) @ W_Q + b_Q
Q = torch.matmul(embeddings_with_pos, query_weight.T) + query_bias
print("Query Matrix (Q):")
print(Q)
print()

# K = (X + PE) @ W_K + b_K
K = torch.matmul(embeddings_with_pos, key_weight.T) + key_bias
print("Key Matrix (K):")
print(K)
print()

# V = (X + PE) @ W_V + b_V
V = torch.matmul(embeddings_with_pos, value_weight.T) + value_bias
print("Value Matrix (V):")
print(V)
print()

Query Matrix (Q):
tensor([[2.0100, 2.2500, 4.4200, 3.7700],
        [2.6298, 3.3022, 4.3137, 3.9820],
        [2.0738, 2.6758, 3.5778, 3.2597]])

Key Matrix (K):
tensor([[3.6100, 5.1200, 4.2600, 4.2900],
        [3.0685, 5.0674, 4.7608, 3.4765],
        [2.5445, 4.0784, 3.9456, 2.8212]])

Value Matrix (V):
tensor([[2.0700, 1.8300, 1.7500, 1.6000],
        [1.7075, 1.7141, 2.0097, 2.1686],
        [1.4318, 1.4562, 1.7210, 1.8713]])



In [96]:
merge_q_k = torch.matmul(Q, K.T) / torch.sqrt(torch.tensor(d_model, dtype=torch.float32))
print("Merged Q and K (QK^T):")
print(merge_q_k)

Merged Q and K (QK^T):
tensor([[26.8893, 25.8592, 21.1831],
        [30.9299, 29.5914, 24.2065],
        [25.2058, 24.1440, 19.7511]])


In [97]:
mask = torch.tensor([
    [0, float('-inf'), float('-inf')],
    [0, 0, float('-inf')],
    [0, 0, 0]
], dtype=torch.float32)

print("Manual Mask:")
print(mask)
print()

Manual Mask:
tensor([[0., -inf, -inf],
        [0., 0., -inf],
        [0., 0., 0.]])



In [98]:
print((merge_q_k + mask).shape)

torch.Size([3, 3])


In [99]:
softmax_result = torch.nn.functional.softmax(merge_q_k + mask, dim=-1)
print(softmax_result)

tensor([[1.0000, 0.0000, 0.0000],
        [0.7922, 0.2078, 0.0000],
        [0.7407, 0.2561, 0.0032]])


In [100]:
attention_output = torch.matmul(softmax_result, V)
print(attention_output)

tensor([[2.0700, 1.8300, 1.7500, 1.6000],
        [1.9947, 1.8059, 1.8040, 1.7181],
        [1.9751, 1.7991, 1.8164, 1.7465]])


In [101]:
layer_norm = torch.nn.LayerNorm(d_model, eps=0)
with torch.no_grad():
    layer_norm.weight.fill_(1.0)  # gamma = 1
    layer_norm.bias.fill_(0.0)    # beta = 0
normalized_output = layer_norm(attention_output + embeddings_with_pos)
print(normalized_output)

tensor([[-0.4887,  0.7924, -1.3943,  1.0906],
        [ 0.8354, -0.7940, -1.1763,  1.1349],
        [ 0.5969, -0.8134, -1.1080,  1.3245]],
       grad_fn=<NativeLayerNormBackward0>)
